In [16]:
from dotenv import load_dotenv

load_dotenv()

True

In [17]:
import requests

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

def get_url(url):
    response = requests.get(url)
    return response.json()

def get_popular_movies():
    """
    Use this when user asks you about what movies are popular these days.
    In this URL, popular movies are listed.
    """
    url = f"{BASE_URL}/movies"
    return get_url(url)

def get_movie_details(id):
    """
    If a user asks which movie corresponds to a given ID,
    use this function to retrieve the movie information.
    """
    url = f"{BASE_URL}/movies/{id}"
    return get_url(url)

def get_movie_credits(id):
    """
    If a user asks who appears in the movie associated with a given ID,
    use this function to retrieve the cast and crew information.
    """
    url = f"{BASE_URL}/movies/{id}/credits"
    return get_url(url)

def get_similar_movies(id):
    """
    If the user asks about the cast and crew of a movie they previously inquired about,
    retrieve the ID of the previously requested movie and use the get_similar_movies.
    """
    url = f"{BASE_URL}/movies/{id}/similar"
    return get_url(url)

def find_movie_id(name):
    """Find movie ID by title from popular movies list.

    Returns one of:
    - {status: "exact", id, title, release_date, candidates: []}
    - {status: "ambiguous", candidates: [{id, title, release_date}, ...]}
    - {status: "not_found", candidates: []}
    """
    movies = get_popular_movies()

    if not isinstance(movies, list):
        return {"status": "not_found", "candidates": []}

    query = (name or "").strip().lower()
    if not query:
        return {"status": "not_found", "candidates": []}

    exact = []
    partial = []

    for movie in movies:
        title = (movie.get("title") or "").strip()
        original_title = (movie.get("original_title") or "").strip()
        title_norm = title.lower()
        original_title_norm = original_title.lower()

        if query == title_norm or query == original_title_norm:
            exact.append(movie)
        elif query in title_norm or query in original_title_norm:
            partial.append(movie)

    if len(exact) == 1:
        matched = exact[0]
        return {
            "status": "exact",
            "id": matched["id"],
            "title": matched.get("title"),
            "release_date": matched.get("release_date"),
            "candidates": []
        }

    if len(exact) > 1:
        sorted_candidates = sorted(exact, key=lambda m: m.get("popularity", 0), reverse=True)
        return {
            "status": "ambiguous",
            "candidates": [
                {
                    "id": c["id"],
                    "title": c.get("title"),
                    "release_date": c.get("release_date")
                }
                for c in sorted_candidates[:5]
            ]
        }

    if partial:
        sorted_candidates = sorted(partial, key=lambda m: m.get("popularity", 0), reverse=True)
        return {
            "status": "ambiguous",
            "candidates": [
                {
                    "id": c["id"],
                    "title": c.get("title"),
                    "release_date": c.get("release_date")
                }
                for c in sorted_candidates[:5]
            ]
        }

    return {"status": "not_found", "candidates": []}

def lookup_movie_id_by_title(title):
    return find_movie_id(title)


In [18]:
tools = [
    {
        "type": "function",
        "name": "get_popular_movies",
        "description": "Use this only when the user explicitly asks what movies are popular or trending these days (current popularity). Do not use this for generic movie recommendations.",
        "parameters": {"type": "object", "properties": {}, "required": []},
    },
    {
        "type": "function",
        "name": "lookup_movie_id_by_title",
        "description": "Use this first when the user gives a movie title instead of a numeric movie ID.",
        "parameters": {
            "type": "object",
            "properties": {
                "title": {
                    "type": "string",
                    "description": "Movie title provided by the user"
                }
            },
            "required": ["title"],
        },
    },
    {
        "type": "function",
        "name": "get_movie_details",
        "description": "Use this when the user explicitly asks for more details about a specific movie",
        "parameters": {
            "type": "object",
            "properties": {"id": {"type": "integer", "description": "Movie ID"}},
            "required": ["id"],
        },
    },
    {
        "type": "function",
        "name": "get_movie_credits",
        "description": "Use this when the user asks about the cast and crew of a movie they previously inquired about",
        "parameters": {
            "type": "object",
            "properties": {"id": {"type": "integer", "description": "Movie ID"}},
            "required": ["id"],
        },
    },
    {
        "type": "function",
        "name": "get_similar_movies",
        "description": "Use this when the user asks for movies similar to one they previously inquired about",
        "parameters": {
            "type": "object",
            "properties": {
                "id": {
                    "type": "integer",
                    "description": "Movie ID"
                }
            },
            "required": ["id"],
        }
    }
]

In [19]:
from openai import OpenAI
import json

client = OpenAI()

messages = []
conversation_state = {
    "pending_candidates": [],
    "pending_intent": None,
    "selected_movie_id": None,
    "selected_movie_title": None,
}

SYSTEM_INSTRUCTION = """
You are a movie assistant that can call tools.

Tool usage rules:
1) Use get_popular_movies only when the user explicitly asks what movies are popular/trending these days.
2) If the user gives a movie title, call lookup_movie_id_by_title first.
3) For details requests, use only get_movie_details.
4) For cast/crew requests, use only get_movie_credits.
5) For similar movie requests, use only get_similar_movies.
6) If lookup status is ambiguous, ask the user to confirm a candidate title.
7) If lookup status is not_found, ask the user to provide a more exact title (optionally with release year).
8) Do not stop after saying you will fetch data; continue calling tools until the final answer is ready.
""".strip()

def get_latest_user_text():
    for msg in reversed(messages):
        if msg.get("role") != "user":
            continue
        for item in msg.get("content", []):
            if item.get("type") == "input_text":
                return item.get("text", "")
    return ""

def detect_intent(user_text):
    text = (user_text or "").lower()
    if any(keyword in text for keyword in ["출연", "캐스트", "배우", "제작진", "감독", "cast", "crew"]):
        return "credits"
    if any(keyword in text for keyword in ["비슷", "유사", "similar"]):
        return "similar"
    if any(keyword in text for keyword in ["자세", "상세", "정보", "줄거리", "평점", "detail", "details"]):
        return "details"
    return None

def is_confirmation(user_text):
    text = (user_text or "").strip().lower()
    if text in {"yes", "yeah", "yep", "ok", "okay"}:
        return True
    if any(keyword in text for keyword in ["맞", "네", "응", "그거", "맞음"]):
        return True
    return False

def choose_tool_by_intent(intent):
    if intent == "credits":
        return "get_movie_credits"
    if intent == "similar":
        return "get_similar_movies"
    return "get_movie_details"

def filter_tools_by_intent(intent):
    allowed_names = {"get_popular_movies", "lookup_movie_id_by_title", choose_tool_by_intent(intent)}
    return [tool for tool in tools if tool.get("name") in allowed_names]

def run_tool(name, args):
    if name == "get_popular_movies":
        return get_popular_movies()

    if name == "lookup_movie_id_by_title":
        title = args.get("title", "")
        result = lookup_movie_id_by_title(title)
        if result.get("status") == "exact":
            conversation_state["selected_movie_id"] = result.get("id")
            conversation_state["selected_movie_title"] = result.get("title")
            conversation_state["pending_candidates"] = []
        elif result.get("status") == "ambiguous":
            conversation_state["pending_candidates"] = result.get("candidates", [])
        else:
            conversation_state["pending_candidates"] = []
        return result

    if name in {"get_movie_details", "get_movie_credits", "get_similar_movies"}:
        movie_id = args.get("id", conversation_state.get("selected_movie_id"))
        if movie_id is None:
            return {"error": "missing_movie_id"}
        movie_id = int(movie_id)
        conversation_state["selected_movie_id"] = movie_id
        if name == "get_movie_details":
            return get_movie_details(movie_id)
        if name == "get_movie_credits":
            return get_movie_credits(movie_id)
        return get_similar_movies(movie_id)

    return {"error": f"Unknown tool: {name}"}

def render_direct_answer(intent, movie_title, movie_id, data):
    if isinstance(data, dict) and data.get("error"):
        return f"영화 ID를 찾았지만 정보를 가져오지 못했습니다. (ID: {movie_id}, 오류: {data['error']})"

    if intent == "credits":
        cast_names = [member.get("name") for member in data.get("cast", [])[:5] if member.get("name")]
        crew_names = [member.get("name") for member in data.get("crew", [])[:5] if member.get("name")]
        return (
            f"확인했습니다. {movie_title} (ID: {movie_id})의 출연진/제작진입니다.\n"
            f"- 출연진(상위): {', '.join(cast_names) if cast_names else '정보 없음'}\n"
            f"- 제작진(상위): {', '.join(crew_names) if crew_names else '정보 없음'}"
        )

    if intent == "similar":
        titles = [movie.get("title") for movie in data[:5] if movie.get("title")] if isinstance(data, list) else []
        return (
            f"확인했습니다. {movie_title} (ID: {movie_id})와 비슷한 영화입니다.\n"
            f"- {', '.join(titles) if titles else '유사 영화 정보 없음'}"
        )

    title = data.get("title", movie_title) if isinstance(data, dict) else movie_title
    release_date = data.get("release_date", "정보 없음") if isinstance(data, dict) else "정보 없음"
    vote = data.get("vote_average", "정보 없음") if isinstance(data, dict) else "정보 없음"
    overview = data.get("overview", "줄거리 정보 없음") if isinstance(data, dict) else "줄거리 정보 없음"
    return (
        f"확인했습니다. {title} (ID: {movie_id}) 상세 정보입니다.\n"
        f"- 개봉일: {release_date}\n"
        f"- 평점: {vote}\n"
        f"- 줄거리: {overview}"
    )

def ask_to_ai():
    latest_user_text = get_latest_user_text()
    detected_intent = detect_intent(latest_user_text)
    current_intent = detected_intent or conversation_state.get("pending_intent") or "details"

    if detected_intent:
        conversation_state["pending_intent"] = detected_intent

    if is_confirmation(latest_user_text):
        candidates = conversation_state.get("pending_candidates", [])
        if len(candidates) == 1:
            candidate = candidates[0]
            movie_id = candidate.get("id")
            movie_title = candidate.get("title", "Unknown")
            intent = conversation_state.get("pending_intent") or "details"

            conversation_state["selected_movie_id"] = movie_id
            conversation_state["selected_movie_title"] = movie_title
            conversation_state["pending_candidates"] = []

            tool_name = choose_tool_by_intent(intent)
            result = run_tool(tool_name, {"id": movie_id})
            conversation_state["pending_intent"] = None
            return render_direct_answer(intent, movie_title, movie_id, result)

    request_input = [
        {
            "role": "system",
            "content": [
                {
                    "type": "input_text",
                    "text": SYSTEM_INSTRUCTION,
                }
            ],
        },
        *messages,
    ]

    allowed_tools = filter_tools_by_intent(current_intent)
    allowed_names = {tool.get("name") for tool in allowed_tools}

    response = client.responses.create(
        model="gpt-4o-mini",
        input=request_input,
        tools=allowed_tools,
    )

    max_steps = 8

    for _ in range(max_steps):
        tool_outputs = []

        for item in response.output:
            if item.type != "function_call":
                continue

            args = json.loads(item.arguments or "{}")
            name = item.name

            print("----------------------")
            print(f"Agent: [{name}({args}) 호출]")
            print("----------------------")

            try:
                if name not in allowed_names:
                    result = {"error": f"disallowed_tool_for_intent: {name}"}
                else:
                    result = run_tool(name, args)
            except Exception as e:
                result = {"error": str(e), "tool": name}

            tool_outputs.append(
                {
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": json.dumps(result, ensure_ascii=False),
                }
            )

        if not tool_outputs:
            return response.output_text

        response = client.responses.create(
            model="gpt-4o-mini",
            previous_response_id=response.id,
            input=tool_outputs,
            tools=allowed_tools,
        )

    return "내부 도구 호출이 반복되어 중단했습니다. 제목을 더 정확히 입력해 주세요."

In [20]:
while True:
    message = input("LLM에게 전달할 메시지를 입력하세요")
    if message == "quit":
        break
    else:
        messages.append({
            "role":"user",
            "content": [
                {
                    "type": "input_text",
                    "text": message
                }
            ]
        })
        print(f"User : {message}")
        print(f"Agent : {ask_to_ai()}")
        print("===================")

User : 지금 인기있는 영화 알려줘
----------------------
Agent: [get_popular_movies({}) 호출]
----------------------
Agent : 지금 인기 있는 영화 몇 편을 소개합니다:

1. **Mercy**
   - **개요**: 가까운 미래, 한 형사가 아내를 살해한 혐의로 재판을 받으며, 자신이 지지했던 AI 판사에게 90분 동안 무죄를 입증해야 하는 이야기입니다.
   - **개봉일**: 2026-01-20
   - **평점**: ★★★★☆ (7.1)
   - ![포스터](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

2. **Shelter**
   - **개요**: 외딴 섬에서 자가 유배 중인 남자가 폭풍 속에서 젊은 소녀를 구출하면서 시작되는 이야기를 다룹니다.
   - **개봉일**: 2026-01-28
   - **평점**: ★★★★☆ (7.0)
   - ![포스터](https://image.tmdb.org/t/p/w780/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg)

3. **The Orphans**
   - **개요**: 어린 시절 서로 나쁜 관계였던 두 친구가 첫사랑을 잃고 사건을 추적하는 이야기입니다.
   - **개봉일**: 2025-08-20
   - **평점**: ★★★☆☆ (6.1)
   - ![포스터](https://image.tmdb.org/t/p/w780/hP7mjZr2SVfjAorlRHTdV1XZmHY.jpg)

4. **The Bluff**
   - **개요**: 한 섬에서 평화로운 삶을 살던 여성이 복수심에 불탄 전 선장과 맞서 싸워야 하는 이야기입니다.
   - **개봉일**: 2026-02-17
   - **평점**: ★★★☆☆ (5.6)
   - ![포스터](https://image.tmdb.org/t/p/w780/sojEzvfxR2DBcDSJyAisX8TWjov.jpg)

